# L3A · Jacobian, vector-Jacobian product, and Hessian in PyTorch

**One idea:** autodiff gives us the Jacobian (local linear map), the VJP
(what backprop actually computes, $\bar x = J^\top\bar y$), and the Hessian
(curvature) — without us deriving anything by hand.

In [1]:
import torch
torch.manual_seed(0)

def f(x):                      # R^2 -> R^2
    return torch.stack([x[0]**2 + x[1],
                        torch.sin(x[0]*x[1])])

x = torch.tensor([1.0, 2.0], requires_grad=True)
J = torch.autograd.functional.jacobian(f, x)
print("Jacobian J =\n", J)

Jacobian J =
 tensor([[ 2.0000,  1.0000],
        [-0.8323, -0.4161]])


## 1. `backward(v)` computes the vector-Jacobian product $v^\top J$
This is exactly what backprop propagates — never the full matrix.

In [2]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = f(x)
v = torch.tensor([1.0, 3.0])
y.backward(v)
print("x.grad (backward) =", x.grad)
print("v^T J   (matmul)  =", v @ J)
print("match:", torch.allclose(x.grad, v @ J, atol=1e-6))

x.grad (backward) = tensor([-0.4969, -0.2484])
v^T J   (matmul)  = tensor([-0.4969, -0.2484])
match: True


## 2. Hessian of a scalar function
For $g(x)=x_0^2 + 3x_1^2 + x_0 x_1$ we expect $H=\begin{bmatrix}2&1\\1&6\end{bmatrix}$.

In [3]:
def g(x):
    return x[0]**2 + 3*x[1]**2 + x[0]*x[1]

H = torch.autograd.functional.hessian(g, x)
print("Hessian H =\n", H)
expected = torch.tensor([[2., 1.], [1., 6.]])
print("matches expected:", torch.allclose(H, expected))

Hessian H =
 tensor([[2., 1.],
        [1., 6.]])
matches expected: True


## Takeaway
- **Jacobian** = local linear map $\mathbb R^n\to\mathbb R^m$.
- **`backward(v)` = VJP** $\;J^\top v$ — the core operation of backpropagation;
  frameworks never materialize the full Jacobian.
- **Hessian** = curvature; here it exactly matches the hand derivation.